# EDA with PySpark

Load train.csv in Spark local mode, profile length distribution, per-subgroup counts and base rates, run the tokenization UDF, write to Parquet.

Local mode here is a demonstration of the API, not a distributed workload — the same job would run unchanged on a cluster.

## 1. Load + schema

In [ ]:
import sys
sys.path.insert(0, "../src")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("jigsaw-eda")
    .master("local[*]")  # local mode: a demonstration of the API, not a
    # distributed workload -- the same job would run unchanged on a cluster.
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

TRAIN_CSV = "../data/raw/train.csv"
df = spark.read.csv(TRAIN_CSV, header=True, inferSchema=True, multiLine=True, escape='"')
print(f"rows: {df.count():,}")
df.printSchema()


## 2. Length distribution (-> max_len choice)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

# Character length over the full data (cheap, exact) ...
char_lengths = df.select(F.length(F.col("comment_text")).alias("char_len")).toPandas()["char_len"]

# ... plus token length under the *actual* tokenizer we'll train with, on a
# sample -- char count is a poor proxy for token count, and max_len is a
# token budget, not a character budget. 50k rows is plenty to estimate
# percentiles precisely; tokenizing all 1.8M here would be wasted compute
# for an EDA plot.
sample_texts = (
    df.select("comment_text").sample(fraction=0.03, seed=42).limit(50_000).toPandas()["comment_text"].fillna("").tolist()
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
token_lengths = np.array([len(tokenizer.encode(t, truncation=False)) for t in sample_texts])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(char_lengths, bins=100, range=(0, 2000))
axes[0].set_title("Character length (full data)")
axes[0].set_xlabel("chars")

axes[1].hist(token_lengths, bins=100, range=(0, 400))
axes[1].axvline(220, color="red", linestyle="--", label="max_len=220")
axes[1].set_title(f"Token length (n={len(sample_texts):,} sample, deberta-v3 tokenizer)")
axes[1].set_xlabel("tokens")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/length_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

for pct in [50, 90, 95, 99, 99.5, 99.9]:
    print(f"p{pct}: char={np.percentile(char_lengths, pct):.0f}  token={np.percentile(token_lengths, pct):.0f}")

coverage_at_220 = (token_lengths <= 220).mean()
print(f"\nfraction of sampled comments fully covered by max_len=220: {coverage_at_220:.4f}")


## 3. Subgroup counts + base rates

In [ ]:
from metrics import IDENTITY_COLUMNS  # the 9 scored subgroups

df = df.withColumn("is_toxic", (F.col("target") >= 0.5).cast("int"))

n_total = df.count()
n_annotated = df.filter(F.col(IDENTITY_COLUMNS[0]).isNotNull()).count()
print(f"rows with identity annotations: {n_annotated:,} / {n_total:,} ({n_annotated/n_total:.1%})")
print(f"overall toxic base rate (target >= 0.5): {df.select(F.mean('is_toxic')).first()[0]:.4f}")

rows = []
for col in IDENTITY_COLUMNS:
    sub = df.filter(F.col(col) >= 0.5)
    n = sub.count()
    rate = sub.select(F.mean("is_toxic")).first()[0] if n > 0 else None
    rows.append((col, n, rate))

import pandas as pd

subgroup_table = pd.DataFrame(rows, columns=["subgroup", "count", "toxic_base_rate"]).sort_values(
    "toxic_base_rate", ascending=False
)
print(subgroup_table.to_string(index=False))

import os
os.makedirs("../reports", exist_ok=True)
subgroup_table.to_csv("../reports/subgroup_base_rates.csv", index=False)

# This table is the first piece of evidence for the bias story: any
# subgroup whose toxic_base_rate sits well above the overall base rate is
# a subgroup where "mentions this identity" and "is toxic" are correlated
# in the training data -- exactly the correlation a model can learn as a
# shortcut instead of learning to recognize actual attacks.


## 4. Tokenization UDF -> Parquet

In [ ]:
import re

from pyspark.sql.types import ArrayType, StringType


def simple_tokenize(text):
    """Lightweight normalization/tokenization UDF -- lowercase, strip
    punctuation, whitespace-split. This is deliberately NOT the tokenizer
    used for the transformer (that's the DeBERTa subword tokenizer,
    applied on the fly in src/train.py); this pass is for fast Spark-side
    text stats and for feeding the TF-IDF baseline from Parquet instead of
    re-parsing the raw CSV every run.
    """
    if text is None:
        return []
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    return text.split()


tokenize_udf = F.udf(simple_tokenize, ArrayType(StringType()))

df_tokenized = df.withColumn("tokens", tokenize_udf(F.col("comment_text")))
df_tokenized = df_tokenized.withColumn("n_tokens", F.size(F.col("tokens")))

OUT_PARQUET = "../data/processed/train_tokenized.parquet"
(
    df_tokenized
    .select("id", "comment_text", "tokens", "n_tokens", "target", *IDENTITY_COLUMNS)
    .write.mode("overwrite")
    .parquet(OUT_PARQUET)
)
print(f"wrote tokenized parquet to {OUT_PARQUET}")

spark.stop()
